In [1]:
%pip install graphframes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 KB 807.4 kB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [1]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
    .appName("urgencenet-graph")
    .master("spark://spark-master:7077")
    .config("spark.jars.packages", "graphframes:graphframes:0.8.4-spark3.5-s_2.12")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate())
print(spark.version, "|", spark.sparkContext.master, "|", spark.sparkContext.defaultParallelism)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c877de08-5890-4f8f-8bcd-93255b0d6e9b;1.0
	confs: [default]
	found graphframes#graphframes;0.8.4-spark3.5-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
:: resolution report :: resolve 2151ms :: artifacts dl 33ms
	:: modules in use:
	graphframes#graphframes;0.8.4-spark3.5-s_2.12 from spark-packages in [default]
	org.slf4j#slf4j-api;1.7.16 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	---------------------------------------------

3.5.3 | spark://spark-master:7077 | 2


In [2]:
from graphframes import GraphFrame

v = spark.createDataFrame([("a",), ("b",)], ["id"])
e = spark.createDataFrame([("a", "b")], ["src", "dst"])
print("GraphFrames OK :", GraphFrame(v, e).inDegrees.collect())

hist = spark.read.parquet("s3a://lake/silver/urgences_history/")
print("Silver historique :", hist.count(), "lignes")
hist.printSchema()
hist.groupBy("status").count().show()

GraphFrames OK : [Row(id='b', inDegree=1)]


26/11/06 17:00:12 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

Silver historique : 49056 lignes
root
 |-- visit_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- facility_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- priority: integer (nullable = true)
 |-- wait_minutes: integer (nullable = true)
 |-- department: string (nullable = true)
 |-- desk_id: string (nullable = true)
 |-- transfer_to: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- city: string (nullable = true)



[Stage 9:>                                                          (0 + 1) / 1]

+------------+-----+
|      status|count|
+------------+-----+
|     ARRIVED|12264|
|     TRIAGED|12264|
| TRANSFERRED| 1185|
|CARE_STARTED|12264|
|  DISCHARGED|11079|
+------------+-----+



In [3]:
FAC = "s3a://lake/bronze/reference/facilities.json"
a = spark.read.json(FAC)
b = spark.read.option("multiLine", "true").json(FAC)
fac = a if ("_corrupt_record" not in a.columns and a.count() >= b.count()) else b
fac = fac.select("facility_id", "name", "city", "hospital_group", "type", "beds")
print("Établissements :", fac.count())
fac.groupBy("hospital_group").count().show()
fac.groupBy("type").count().show()

Établissements : 40


+--------------+-----+
|hospital_group|count|
+--------------+-----+
|      GHT-Nord|   10|
|GHT-Atlantique|   20|
|       GHT-Sud|   10|
+--------------+-----+



[Stage 26:>                                                         (0 + 1) / 1]

+----------------+-----+
|            type|count|
+----------------+-----+
|             CHU|    4|
|    CENTRE_SANTE|   12|
|        CLINIQUE|   16|
|HOPITAL_REGIONAL|    8|
+----------------+-----+



In [4]:
from pyspark.sql.types import StringType, TimestampType

# Sommets
v_fac = fac.select(F.col("facility_id").alias("id"), F.lit("FACILITY").alias("type"),
                   "hospital_group", F.col("name"), "city")

dep = (hist.filter(F.col("department").isNotNull())
       .select("facility_id", "department").distinct()
       .withColumn("id", F.concat(F.lit("D-"), "facility_id", F.lit("-"), "department")))

v_dep = (dep.join(fac.select("facility_id", "hospital_group", "city"), "facility_id")
         .select("id", F.lit("DEPARTMENT").alias("type"), "hospital_group",
                 F.col("department").alias("name"), "city"))

first_arr = (hist.filter(F.col("status") == "ARRIVED")
             .groupBy("patient_id")
             .agg(F.min(F.struct("event_time", "facility_id")).alias("m"))
             .select("patient_id", F.col("m.facility_id").alias("facility_id")))

v_pat = (first_arr.join(fac.select("facility_id", "hospital_group", "city"), "facility_id", "left")
         .select(F.col("patient_id").alias("id"), F.lit("PATIENT").alias("type"),
                 "hospital_group", F.lit(None).cast(StringType()).alias("name"), "city"))

vertices = v_pat.unionByName(v_fac).unionByName(v_dep).cache()

# Arêtes (schéma commun)
def edges_of(df, src, dst, rel):
    return df.select(src.alias("src"), dst.alias("dst"), F.lit(rel).alias("relationship"),
                     "visit_id", "patient_id", "event_time")

dep_id = F.concat(F.lit("D-"), F.col("facility_id"), F.lit("-"), F.col("department"))
admitted = edges_of(hist.filter("status = 'ARRIVED'"), F.col("patient_id"), F.col("facility_id"), "ADMITTED_AT")
treated  = edges_of(hist.filter("status = 'CARE_STARTED' AND department IS NOT NULL"),
                    F.col("patient_id"), dep_id, "TREATED_IN")
transf   = edges_of(hist.filter("status = 'TRANSFERRED'"), F.col("facility_id"), F.col("transfer_to"), "TRANSFERRED_TO")
has_dep  = (dep.select(F.col("facility_id").alias("src"), F.col("id").alias("dst"),
                       F.lit("HAS_DEPARTMENT").alias("relationship"),
                       F.lit(None).cast(StringType()).alias("visit_id"),
                       F.lit(None).cast(StringType()).alias("patient_id"),
                       F.lit(None).cast(TimestampType()).alias("event_time")))

edges = admitted.unionByName(treated).unionByName(transf).unionByName(has_dep).cache()

print("Sommets par type et groupe :")
vertices.groupBy("type", "hospital_group").count().orderBy("type", "hospital_group").show(20)
print("Arêtes par relation :")
edges.groupBy("relationship").count().show()

# Contrôles d'intégrité
print("Identifiants en collision :", vertices.groupBy("id").count().filter("count > 1").count())
print("Arêtes avec src inconnu :", edges.join(vertices, edges.src == vertices.id, "left_anti").count())
print("Arêtes avec dst inconnu :", edges.join(vertices, edges.dst == vertices.id, "left_anti").count())
print("Sommets sans hospital_group :", vertices.filter("hospital_group IS NULL").count())

Sommets par type et groupe :


+----------+--------------+-----+
|      type|hospital_group|count|
+----------+--------------+-----+
|DEPARTMENT|GHT-Atlantique|  102|
|DEPARTMENT|      GHT-Nord|   51|
|DEPARTMENT|       GHT-Sud|   51|
|  FACILITY|GHT-Atlantique|   20|
|  FACILITY|      GHT-Nord|   10|
|  FACILITY|       GHT-Sud|   10|
|   PATIENT|GHT-Atlantique| 3024|
|   PATIENT|      GHT-Nord| 1503|
|   PATIENT|       GHT-Sud| 1623|
+----------+--------------+-----+

Arêtes par relation :


+--------------+-----+
|  relationship|count|
+--------------+-----+
|   ADMITTED_AT|12264|
|    TREATED_IN|12264|
|TRANSFERRED_TO| 1185|
|HAS_DEPARTMENT|  204|
+--------------+-----+



Identifiants en collision : 0


Arêtes avec src inconnu : 0


Arêtes avec dst inconnu : 0


[Stage 78:=================================================>      (15 + 2) / 17]

Sommets sans hospital_group : 0


In [5]:
from graphframes import GraphFrame
g = GraphFrame(vertices, edges)
print("GraphFrame :", g.vertices.count(), "sommets,", g.edges.count(), "arêtes")

# Sommets par groupe et type (déjà vu) + arêtes par groupe de la source et par relation
(g.edges.join(g.vertices.select(F.col("id").alias("src"), F.col("hospital_group").alias("src_group")), "src")
   .groupBy("src_group", "relationship").count()
   .orderBy("src_group", "relationship").show(20))

# Transferts enrichis : groupe/type de la source et de la destination
tr = (edges.filter("relationship = 'TRANSFERRED_TO'")
      .join(fac.select(F.col("facility_id").alias("src"), F.col("hospital_group").alias("src_group"),
                       F.col("type").alias("src_type")), "src")
      .join(fac.select(F.col("facility_id").alias("dst"), F.col("hospital_group").alias("dst_group"),
                       F.col("type").alias("dst_type")), "dst")).cache()
print("Transferts retrouvés (attendu 1185) :", tr.count())

# Intra-groupe contre inter-groupes
(tr.withColumn("portee", F.when(F.col("src_group") == F.col("dst_group"), "intra-groupe").otherwise("inter-groupes"))
   .groupBy("src_group", "portee").count().orderBy("src_group", "portee").show())

# Transferts inter-groupes : par groupe source, groupe destination et type de destination
(tr.filter("src_group <> dst_group")
   .groupBy("src_group", "dst_group", "dst_type").count()
   .orderBy("src_group", "dst_group", "dst_type").show(30))

# Règle de l'énoncé : un transfert inter-groupes n'est autorisé que vers un CHU
print("Violations (inter-groupes vers un non-CHU) :", tr.filter("src_group <> dst_group AND dst_type <> 'CHU'").count())
print("Auto-transferts (src = dst) :", tr.filter("src = dst").count())

GraphFrame : 6394 sommets, 25917 arêtes


+--------------+--------------+-----+
|     src_group|  relationship|count|
+--------------+--------------+-----+
|GHT-Atlantique|   ADMITTED_AT| 6118|
|GHT-Atlantique|HAS_DEPARTMENT|  102|
|GHT-Atlantique|TRANSFERRED_TO|  573|
|GHT-Atlantique|    TREATED_IN| 6118|
|      GHT-Nord|   ADMITTED_AT| 2945|
|      GHT-Nord|HAS_DEPARTMENT|   51|
|      GHT-Nord|TRANSFERRED_TO|  320|
|      GHT-Nord|    TREATED_IN| 2945|
|       GHT-Sud|   ADMITTED_AT| 3201|
|       GHT-Sud|HAS_DEPARTMENT|   51|
|       GHT-Sud|TRANSFERRED_TO|  292|
|       GHT-Sud|    TREATED_IN| 3201|
+--------------+--------------+-----+



Transferts retrouvés (attendu 1185) : 1185


+--------------+-------------+-----+
|     src_group|       portee|count|
+--------------+-------------+-----+
|GHT-Atlantique|inter-groupes|   75|
|GHT-Atlantique| intra-groupe|  498|
|      GHT-Nord|inter-groupes|   40|
|      GHT-Nord| intra-groupe|  280|
|       GHT-Sud|inter-groupes|   23|
|       GHT-Sud| intra-groupe|  269|
+--------------+-------------+-----+



+--------------+--------------+--------+-----+
|     src_group|     dst_group|dst_type|count|
+--------------+--------------+--------+-----+
|GHT-Atlantique|      GHT-Nord|     CHU|   40|
|GHT-Atlantique|       GHT-Sud|     CHU|   34|
|GHT-Atlantique|       GHT-Sud|CLINIQUE|    1|
|      GHT-Nord|GHT-Atlantique|     CHU|   29|
|      GHT-Nord|       GHT-Sud|     CHU|   11|
|       GHT-Sud|GHT-Atlantique|     CHU|   14|
|       GHT-Sud|      GHT-Nord|     CHU|    9|
+--------------+--------------+--------+-----+



Violations (inter-groupes vers un non-CHU) : 1


[Stage 127:=================================================>     (18 + 2) / 20]

Auto-transferts (src = dst) : 0


In [6]:
# 0) Détail de l'unique violation
(tr.filter("src_group <> dst_group AND dst_type <> 'CHU'")
   .select("visit_id","patient_id","src","src_group","dst","dst_group","dst_type","event_time")
   .show(truncate=False))

# Graphe établissement -> établissement
fv = vertices.filter("type = 'FACILITY'")
fe = edges.filter("relationship = 'TRANSFERRED_TO'").select("src","dst","visit_id","patient_id","event_time")
gf = GraphFrame(fv, fe)

# 1) Chaînes A -> B -> C (même patient, ordre chronologique, 3 établissements distincts)
chains = (gf.find("(a)-[e1]->(b); (b)-[e2]->(c)")
            .filter("e1.patient_id = e2.patient_id AND e1.event_time < e2.event_time "
                    "AND a.id <> c.id AND a.id <> b.id AND b.id <> c.id"))
print("Chaînes A->B->C (même patient) :", chains.count())
chains.select("a.id","b.id","c.id","e1.patient_id","e1.event_time","e2.event_time").show(10, truncate=False)

# 2) Boucles A -> B -> A (même patient)
loops = (gf.find("(a)-[e1]->(b); (b)-[e2]->(a)")
           .filter("e1.patient_id = e2.patient_id AND e1.event_time < e2.event_time"))
print("Boucles A->B->A (même patient) :", loops.count())
loops.select("a.id","b.id","e1.patient_id","e1.event_time","e2.event_time").show(10, truncate=False)

# 3) Patients admis dans plusieurs établissements (motif sur le graphe complet)
multi = (g.find("(p)-[e1]->(f1); (p)-[e2]->(f2)")
           .filter("e1.relationship = 'ADMITTED_AT' AND e2.relationship = 'ADMITTED_AT' AND f1.id < f2.id"))
print("Patients admis dans >= 2 établissements :", multi.select("p.id").distinct().count())
multi.select("p.id","f1.id","f2.id").show(10, truncate=False)

# 4) Degrés du graphe établissement -> établissement
deg = (fv.select("id","hospital_group","name","city")
         .join(gf.outDegrees, "id", "left").join(gf.inDegrees, "id", "left").fillna(0, ["outDegree","inDegree"]))
deg.orderBy(F.desc("outDegree")).show(8, truncate=False)
deg.orderBy(F.desc("inDegree")).show(8, truncate=False)
deg.select(F.round(F.avg("outDegree"),1).alias("moy_out"), F.round(F.stddev("outDegree"),1).alias("ecart_out"),
           F.round(F.avg("inDegree"),1).alias("moy_in"), F.round(F.stddev("inDegree"),1).alias("ecart_in")).show()

+---------+----------+-----+--------------+-----+---------+--------+-------------------+
|visit_id |patient_id|src  |src_group     |dst  |dst_group|dst_type|event_time         |
+---------+----------+-----+--------------+-----+---------+--------+-------------------+
|V-0005629|P-04788   |F-012|GHT-Atlantique|F-026|GHT-Sud  |CLINIQUE|2026-10-18 17:03:45|
+---------+----------+-----+--------------+-----+---------+--------+-------------------+



Chaînes A->B->C (même patient) : 159


+-----+-----+-----+----------+-------------------+-------------------+
|id   |id   |id   |patient_id|event_time         |event_time         |
+-----+-----+-----+----------+-------------------+-------------------+
|F-021|F-022|F-023|P-03264   |2026-10-19 19:00:27|2026-10-19 19:32:14|
|F-022|F-023|F-021|P-03264   |2026-10-19 19:32:14|2026-10-19 20:24:37|
|F-021|F-024|F-022|P-03993   |2026-10-06 23:22:21|2026-10-07 00:39:49|
|F-023|F-031|F-036|P-00077   |2026-10-22 18:03:24|2026-10-22 19:59:43|
|F-027|F-011|F-002|P-05499   |2026-10-18 18:27:52|2026-10-18 21:21:37|
|F-029|F-022|F-021|P-03758   |2026-10-25 18:18:30|2026-10-25 20:05:07|
|F-021|F-023|F-022|P-03002   |2026-10-24 21:19:07|2026-10-24 22:58:17|
|F-017|F-007|F-011|P-00098   |2026-10-29 23:46:56|2026-10-30 00:32:03|
|F-011|F-014|F-001|P-03895   |2026-11-01 20:21:31|2026-11-01 21:26:25|
|F-014|F-001|F-013|P-03895   |2026-11-01 21:26:25|2026-11-01 22:21:39|
+-----+-----+-----+----------+-------------------+-------------------+
only s

[Stage 169:=================================================>     (18 + 2) / 20]

Boucles A->B->A (même patient) : 184


+-----+-----+----------+-------------------+-------------------+
|id   |id   |patient_id|event_time         |event_time         |
+-----+-----+----------+-------------------+-------------------+
|F-021|F-022|P-01570   |2026-10-09 11:51:34|2026-10-09 12:56:20|
|F-024|F-022|P-00129   |2026-10-11 00:29:24|2026-10-11 04:09:44|
|F-024|F-022|P-00129   |2026-10-11 00:29:24|2026-10-11 01:47:43|
|F-022|F-024|P-00129   |2026-10-11 01:47:43|2026-10-11 03:13:39|
|F-024|F-022|P-00129   |2026-10-11 03:13:39|2026-10-11 04:09:44|
|F-029|F-021|P-03251   |2026-10-25 05:57:07|2026-10-25 10:17:27|
|F-029|F-021|P-03251   |2026-10-25 05:57:07|2026-10-25 07:53:50|
|F-021|F-029|P-03251   |2026-10-25 07:53:50|2026-10-25 09:11:00|
|F-029|F-021|P-03251   |2026-10-25 09:11:00|2026-10-25 10:17:27|
|F-022|F-021|P-03993   |2026-10-07 02:05:16|2026-10-07 03:13:31|
+-----+-----+----------+-------------------+-------------------+
only showing top 10 rows



... (2 outputs left out)

+-----+--------------+------------------------------+----------+---------+--------+
|id   |hospital_group|name                          |city      |outDegree|inDegree|
+-----+--------------+------------------------------+----------+---------+--------+
|F-011|GHT-Atlantique|CHU Rabat                     |Rabat     |111      |102     |
|F-031|GHT-Nord      |CHU Tanger                    |Tanger    |98       |122     |
|F-021|GHT-Sud       |CHU Marrakech                 |Marrakech |97       |130     |
|F-001|GHT-Atlantique|CHU Casablanca                |Casablanca|87       |97      |
|F-023|GHT-Sud       |Hôpital Régional Sud Marrakech|Marrakech |62       |55      |
|F-032|GHT-Nord      |Hôpital Régional Nord Tanger  |Tanger    |58       |58      |
|F-013|GHT-Atlantique|Hôpital Régional Sud Rabat    |Rabat     |53       |58      |
|F-012|GHT-Atlantique|Hôpital Régional Nord Rabat   |Rabat     |51       |48      |
+-----+--------------+------------------------------+----------+---------+--

+-----+--------------+------------------------------+----------+---------+--------+
|id   |hospital_group|name                          |city      |outDegree|inDegree|
+-----+--------------+------------------------------+----------+---------+--------+
|F-021|GHT-Sud       |CHU Marrakech                 |Marrakech |97       |130     |
|F-031|GHT-Nord      |CHU Tanger                    |Tanger    |98       |122     |
|F-011|GHT-Atlantique|CHU Rabat                     |Rabat     |111      |102     |
|F-001|GHT-Atlantique|CHU Casablanca                |Casablanca|87       |97      |
|F-013|GHT-Atlantique|Hôpital Régional Sud Rabat    |Rabat     |53       |58      |
|F-032|GHT-Nord      |Hôpital Régional Nord Tanger  |Tanger    |58       |58      |
|F-023|GHT-Sud       |Hôpital Régional Sud Marrakech|Marrakech |62       |55      |
|F-033|GHT-Nord      |Hôpital Régional Sud Tanger   |Tanger    |48       |55      |
+-----+--------------+------------------------------+----------+---------+--

[Stage 242:=============================================>         (14 + 3) / 17]

+-------+---------+------+--------+
|moy_out|ecart_out|moy_in|ecart_in|
+-------+---------+------+--------+
|   29.6|     28.3|  29.6|    33.5|
+-------+---------+------+--------+



In [7]:
# A) Transferts par patient (seuil >= 4 du point 6)
tp = fe.groupBy("patient_id").agg(F.count("*").alias("nb_transferts"))
tp.groupBy("nb_transferts").count().orderBy("nb_transferts").show()
print("Patients avec >= 4 transferts :", tp.filter("nb_transferts >= 4").count())
tp.orderBy(F.desc("nb_transferts")).show(5)

# B) Chaînes et boucles : en patients distincts
print("Patients avec une chaîne A->B->C :", chains.select("e1.patient_id").distinct().count())
print("Patients avec une boucle A->B->A :", loops.select("e1.patient_id").distinct().count())

# C) Composantes connexes (graphe établissement -> établissement, non orienté pour ce calcul)
spark.sparkContext.setCheckpointDir("s3a://lake/checkpoints/graphframes/")
cc = gf.connectedComponents()
(cc.groupBy("component").agg(F.count("*").alias("nb_etab"), F.collect_set("hospital_group").alias("groupes"))
   .show(truncate=False))

# D) Idem en ne gardant que les transferts intra-groupe
fg = fv.select("id", "hospital_group")
fe_intra = (fe.join(fg.withColumnRenamed("id", "src").withColumnRenamed("hospital_group", "g1"), "src")
              .join(fg.withColumnRenamed("id", "dst").withColumnRenamed("hospital_group", "g2"), "dst")
              .filter("g1 = g2").select("src", "dst"))
cc2 = GraphFrame(fv, fe_intra).connectedComponents()
cc2.groupBy("hospital_group", "component").count().orderBy("hospital_group", "component").show(20)

+-------------+-----+
|nb_transferts|count|
+-------------+-----+
|            1|  695|
|            2|   77|
|            3|    5|
|            4|   74|
|            5|    5|
+-------------+-----+



Patients avec >= 4 transferts : 79


+----------+-------------+
|patient_id|nb_transferts|
+----------+-------------+
|   P-02865|            5|
|   P-00304|            5|
|   P-03264|            5|
|   P-01570|            5|
|   P-00891|            5|
+----------+-------------+
only showing top 5 rows



Patients avec une chaîne A->B->C : 91


Patients avec une boucle A->B->A : 57


Py4JJavaError: An error occurred while calling o590.run.
: java.lang.OutOfMemoryError: Java heap space
	at java.base/java.util.Arrays.copyOfRange(Arrays.java:3822)
	at java.base/java.lang.StringLatin1.newString(StringLatin1.java:769)
	at java.base/java.lang.StringBuilder.toString(StringBuilder.java:453)
	at org.apache.spark.sql.catalyst.util.StringConcat.toString(StringUtils.scala:64)
	at org.apache.spark.sql.catalyst.util.StringUtils$PlanStringConcat.toString(StringUtils.scala:152)
	at org.apache.spark.sql.execution.QueryExecution.explainString(QueryExecution.scala:254)
	at org.apache.spark.sql.execution.SQLExecution$.$anonfun$withNewExecutionId$6(SQLExecution.scala:117)
	at org.apache.spark.sql.execution.SQLExecution$$$Lambda$1903/0x00007bf4fcc51528.apply(Unknown Source)
	at org.apache.spark.sql.execution.SQLExecution$.withSQLConfPropagated(SQLExecution.scala:201)
	at org.apache.spark.sql.execution.SQLExecution$.$anonfun$withNewExecutionId$1(SQLExecution.scala:108)
	at org.apache.spa

In [8]:
# Graphe réduit : on coupe le lineage en repartant de listes Python
fac_rows = fv.select("id", "hospital_group").collect()
grp = {r.id: r.hospital_group for r in fac_rows}
fv_small = spark.createDataFrame([(r.id, r.hospital_group) for r in fac_rows], ["id", "hospital_group"])

pairs = [(r.src, r.dst) for r in fe.select("src", "dst").distinct().collect()]
print("Paires (src,dst) distinctes :", len(pairs))
fe_small = spark.createDataFrame(pairs, ["src", "dst"])

# C) Composantes connexes, graphe complet (arêtes traitées comme non orientées)
cc = GraphFrame(fv_small, fe_small).connectedComponents()
(cc.groupBy("component").agg(F.count("*").alias("nb_etab"), F.collect_set("hospital_group").alias("groupes"))
   .show(truncate=False))

# D) Composantes connexes en ne gardant que les transferts intra-groupe
pairs_intra = [(s, d) for (s, d) in pairs if grp[s] == grp[d]]
print("Paires intra-groupe :", len(pairs_intra))
cc2 = GraphFrame(fv_small, spark.createDataFrame(pairs_intra, ["src", "dst"])).connectedComponents()
cc2.groupBy("hospital_group", "component").count().orderBy("hospital_group", "component").show(20)

Paires (src,dst) distinctes : 325


Py4JJavaError: An error occurred while calling o654.run.
: java.lang.OutOfMemoryError: Java heap space
	at java.base/java.util.Arrays.copyOfRange(Arrays.java:3822)
	at java.base/java.lang.StringLatin1.newString(StringLatin1.java:769)
	at java.base/java.lang.StringBuilder.toString(StringBuilder.java:453)
	at org.apache.spark.sql.catalyst.util.StringConcat.toString(StringUtils.scala:64)
	at org.apache.spark.sql.catalyst.util.StringUtils$PlanStringConcat.toString(StringUtils.scala:152)
	at org.apache.spark.sql.execution.QueryExecution.explainString(QueryExecution.scala:254)
	at org.apache.spark.sql.execution.adaptive.AdaptiveSparkPlanExec.onUpdatePlan(AdaptiveSparkPlanExec.scala:777)
	at org.apache.spark.sql.execution.adaptive.AdaptiveSparkPlanExec.$anonfun$getFinalPhysicalPlan$2(AdaptiveSparkPlanExec.scala:285)
	at org.apache.spark.sql.execution.adaptive.AdaptiveSparkPlanExec$$Lambda$2327/0x00007bf4fcd78890.apply$mcVJ$sp(Unknown Source)
	at scala.runtime.java8.JFunction1$mcVJ$sp.apply(JF

In [9]:
def components(nodes, prs):
    parent = {n: n for n in nodes}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for s, d in prs:
        parent[find(s)] = find(d)
    comp = {}
    for n in nodes:
        comp.setdefault(find(n), []).append(n)
    return list(comp.values())

nodes = list(grp)
c_all = components(nodes, pairs)
print("Graphe complet :", len(c_all), "composante(s), tailles", sorted(len(c) for c in c_all))

for gname in sorted(set(grp.values())):
    ns = [n for n in nodes if grp[n] == gname]
    ps = [(s, d) for s, d in pairs if grp[s] == gname and grp[d] == gname]
    cs = components(ns, ps)
    print(f"{gname} (intra-groupe) :", len(cs), "composante(s), tailles", sorted(len(c) for c in cs))

Graphe complet : 1 composante(s), tailles [40]
GHT-Atlantique (intra-groupe) : 1 composante(s), tailles [20]
GHT-Nord (intra-groupe) : 1 composante(s), tailles [10]
GHT-Sud (intra-groupe) : 1 composante(s), tailles [10]


In [10]:
spark.sparkContext.setCheckpointDir("s3a://lake/checkpoints/graphframes/")
cc = GraphFrame(fv_small, fe_small).connectedComponents(algorithm="graphx")
(cc.groupBy("component")
   .agg(F.count("*").alias("nb_etab"), F.collect_set("hospital_group").alias("groupes"))
   .show(truncate=False))

26/11/06 17:22:10 WARN CacheManager: Asked to cache already cached data.
[Stage 849:>                                                        (0 + 1) / 1]

+---------+-------+-----------------------------------+
|component|nb_etab|groupes                            |
+---------+-------+-----------------------------------+
|0        |40     |[GHT-Nord, GHT-Sud, GHT-Atlantique]|
+---------+-------+-----------------------------------+



In [11]:
base = "s3a://lake/gold/graph_analysis/"

# Tables du graphe (entrées de GraphX)
vertices.write.mode("overwrite").parquet(base + "input/vertices")
edges.write.mode("overwrite").parquet(base + "input/edges")

# Graphe pondéré des transferts entre établissements (poids = nombre de transferts)
wt = (tr.groupBy("src", "dst", "src_group", "dst_group").count()
        .withColumnRenamed("count", "weight"))
wt.write.mode("overwrite").parquet(base + "input/transfer_weights")

# Résultats GraphFrames, séparés par hospital_group
deg.write.mode("overwrite").partitionBy("hospital_group").parquet(base + "graphframes/degrees")
cc.write.mode("overwrite").partitionBy("hospital_group").parquet(base + "graphframes/components")

# Relecture et contrôles
rb = lambda p: spark.read.parquet(base + p)
print("vertices (attendu 6394) :", rb("input/vertices").count())
print("edges (attendu 25917)   :", rb("input/edges").count())
print("transfer_weights (attendu 325) :", rb("input/transfer_weights").count(),
      "| somme des poids (attendu 1185) :", rb("input/transfer_weights").agg(F.sum("weight")).first()[0])
print("degrees (attendu 40)    :", rb("graphframes/degrees").count())
print("components (attendu 40) :", rb("graphframes/components").count())

vertices (attendu 6394) : 6394


edges (attendu 25917)   : 25917


transfer_weights (attendu 325) : 325 | somme des poids (attendu 1185) : 1185
degrees (attendu 40)    : 40
components (attendu 40) : 40


26/11/06 20:23:23 ERROR TaskSchedulerImpl: Lost executor 0 on 172.21.0.6: worker lost: Not receiving heartbeat for 60 seconds
26/11/06 20:23:23 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_670_1 !
26/11/06 20:23:23 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_872_6 !
26/11/06 20:23:23 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_97_10 !
26/11/06 20:23:23 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_713_6 !
26/11/06 20:23:23 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_590_2 !
26/11/06 20:23:23 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_124_10 !
26/11/06 20:23:23 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_713_3 !
26/11/06 20:23:23 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_704_2 !
26/11/06 20:23:23 WARN BlockManagerMasterEndpoint: No more replicas available for rdd_704_5 !
26/11/06 20:23:23 WARN Bloc

In [1]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
    .appName("UrgenceNet-Partie6-Comparaison")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "4")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print(spark.version, spark.sparkContext.defaultParallelism)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/06 21:40:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


3.5.3 2


In [2]:
base = "s3a://lake/gold/graph_analysis/"
gf = spark.read.parquet(base + "graphframes/degrees")
gx = spark.read.parquet(base + "graphx/metrics")
gf.printSchema(); gf.show(5, truncate=False)
gx.printSchema(); gx.show(5, truncate=False)

26/11/06 21:40:55 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- outDegree: integer (nullable = true)
 |-- inDegree: integer (nullable = true)
 |-- hospital_group: string (nullable = true)



+-----+--------------------------------+----------+---------+--------+--------------+
|id   |name                            |city      |outDegree|inDegree|hospital_group|
+-----+--------------------------------+----------+---------+--------+--------------+
|F-001|CHU Casablanca                  |Casablanca|87       |97      |GHT-Atlantique|
|F-002|Hôpital Régional Nord Casablanca|Casablanca|36       |44      |GHT-Atlantique|
|F-003|Hôpital Régional Sud Casablanca |Casablanca|48       |47      |GHT-Atlantique|
|F-004|Clinique Al Amal Casablanca     |Casablanca|11       |11      |GHT-Atlantique|
|F-005|Clinique Ennour Casablanca      |Casablanca|14       |14      |GHT-Atlantique|
+-----+--------------------------------+----------+---------+--------+--------------+
only showing top 5 rows

root
 |-- facility_id: string (nullable = true)
 |-- degree: integer (nullable = true)
 |-- weighted_degree: long (nullable = true)
 |-- component: long (nullable = true)
 |-- pagerank: double (nullabl

[Stage 3:>                                                          (0 + 1) / 1]

+-----------+------+---------------+---------+------------------+--------------+
|facility_id|degree|weighted_degree|component|pagerank          |hospital_group|
+-----------+------+---------------+---------+------------------+--------------+
|F-021      |18    |173            |21       |1.7618392910584957|GHT-Sud       |
|F-022      |13    |94             |21       |1.2915933796183454|GHT-Sud       |
|F-023      |17    |112            |21       |1.7430374872228505|GHT-Sud       |
|F-024      |9     |36             |21       |0.6845694552857888|GHT-Sud       |
|F-025      |8     |34             |21       |0.9806077591139766|GHT-Sud       |
+-----------+------+---------------+---------+------------------+--------------+
only showing top 5 rows



In [3]:
wt = spark.read.parquet(base + "input/transfer_weights")
intra = wt.filter("src_group = dst_group")

ends = (intra.select(F.col("src").alias("facility_id"), "weight")
        .unionByName(intra.select(F.col("dst").alias("facility_id"), "weight")))
ref = ends.groupBy("facility_id").agg(F.count("*").alias("degree_ref"),
                                      F.sum("weight").alias("wdeg_ref"))

cmp = (gf.select(F.col("id").alias("facility_id"), "hospital_group",
                 (F.col("outDegree") + F.col("inDegree")).alias("gf_total"))
       .join(gx.select("facility_id", "degree", "weighted_degree"), "facility_id", "left")
       .join(ref, "facility_id", "left")
       .withColumn("inter_groupe", F.col("gf_total") - F.col("weighted_degree")))

print("Établissements comparés :", cmp.count())
print("GraphX degree != référence          :", cmp.filter("degree != degree_ref").count())
print("GraphX weighted_degree != référence :", cmp.filter("weighted_degree != wdeg_ref").count())
cmp.agg(F.sum("gf_total").alias("somme_GF (2 x transferts)"),
        F.sum("weighted_degree").alias("somme_GX (2 x intra)"),
        F.sum("inter_groupe").alias("somme_ecart")).show()
cmp.orderBy("facility_id").show(8, truncate=False)

Établissements comparés : 40


GraphX degree != référence          : 0


GraphX weighted_degree != référence : 0


+-------------------------+--------------------+-----------+
|somme_GF (2 x transferts)|somme_GX (2 x intra)|somme_ecart|
+-------------------------+--------------------+-----------+
|                     2370|                2094|        276|
+-------------------------+--------------------+-----------+



+-----------+--------------+--------+------+---------------+----------+--------+------------+
|facility_id|hospital_group|gf_total|degree|weighted_degree|degree_ref|wdeg_ref|inter_groupe|
+-----------+--------------+--------+------+---------------+----------+--------+------------+
|F-001      |GHT-Atlantique|184     |26    |151            |26        |151     |33          |
|F-002      |GHT-Atlantique|80      |26    |77             |26        |77      |3           |
|F-003      |GHT-Atlantique|95      |24    |91             |24        |91      |4           |
|F-004      |GHT-Atlantique|22      |13    |20             |13        |20      |2           |
|F-005      |GHT-Atlantique|28      |14    |26             |14        |26      |2           |
|F-006      |GHT-Atlantique|35      |18    |34             |18        |34      |1           |
|F-007      |GHT-Atlantique|44      |18    |42             |18        |42      |2           |
|F-008      |GHT-Atlantique|4       |3     |4              |

In [4]:
edges_in = spark.read.parquet(base + "input/edges")
verts = spark.read.parquet(base + "input/vertices")
tr = edges_in.filter("relationship = 'TRANSFERRED_TO'")

fac_ids = verts.filter("type = 'FACILITY'").select(F.col("id").alias("fid"))
print("Transferts :", tr.count())
print("Vers un établissement inexistant :",
      tr.join(fac_ids, tr.dst == fac_ids.fid, "left_anti").count())
print("D'un établissement vers lui-même :", tr.filter("src = dst").count())
print("Patients avec >= 4 transferts :",
      tr.groupBy("patient_id").count().filter("count >= 4").count())

Transferts : 1185


Vers un établissement inexistant : 0


D'un établissement vers lui-même : 0


[Stage 46:==========================================================(1 + 0) / 1]

Patients avec >= 4 transferts : 79


In [5]:
by_p = (tr.groupBy("patient_id")
          .agg(F.count("*").alias("nb_transferts"),
               F.collect_list(F.struct("event_time", "src", "dst")).alias("parcours"))
          .filter("nb_transferts >= 4"))

by_p = by_p.withColumn("chemin", F.concat_ws(" > ",
        F.transform(F.array_sort("parcours"), lambda x: F.concat(x["src"], F.lit("→"), x["dst"]))))

print("Répartition du nombre de transferts par patient (>= 4) :")
by_p.groupBy("nb_transferts").count().orderBy("nb_transferts").show()
by_p.select("patient_id", "nb_transferts", "chemin").orderBy(F.desc("nb_transferts")).show(8, truncate=False)

Répartition du nombre de transferts par patient (>= 4) :


+-------------+-----+
|nb_transferts|count|
+-------------+-----+
|            4|   74|
|            5|    5|
+-------------+-----+



+----------+-------------+-------------------------------------------------------------------+
|patient_id|nb_transferts|chemin                                                             |
+----------+-------------+-------------------------------------------------------------------+
|P-00304   |5            |F-040→F-031 > F-031→F-040 > F-040→F-031 > F-031→F-040 > F-025→F-021|
|P-00891   |5            |F-013→F-001 > F-016→F-011 > F-011→F-016 > F-016→F-011 > F-011→F-016|
|P-01570   |5            |F-022→F-021 > F-021→F-022 > F-022→F-021 > F-021→F-022 > F-023→F-021|
|P-02865   |5            |F-011→F-031 > F-001→F-003 > F-003→F-015 > F-015→F-009 > F-009→F-016|
|P-03264   |5            |F-028→F-021 > F-021→F-022 > F-022→F-023 > F-023→F-021 > F-001→F-011|
|P-00021   |4            |F-011→F-003 > F-003→F-012 > F-012→F-020 > F-020→F-006              |
|P-00072   |4            |F-040→F-021 > F-021→F-011 > F-011→F-001 > F-001→F-012              |
|P-00098   |4            |F-017→F-007 > F-007→F-01

In [6]:
w = Window = None
from pyspark.sql import Window
w = Window.partitionBy("patient_id").orderBy("event_time")

seq = (tr.select("patient_id", "event_time", "src", "dst")
         .withColumn("prev_src", F.lag("src").over(w))
         .withColumn("prev_dst", F.lag("dst").over(w)))

# boucle : le transfert courant repart de l'ancienne destination vers l'ancienne source
loops = seq.filter((F.col("src") == F.col("prev_dst")) & (F.col("dst") == F.col("prev_src")))

print("Transferts qui referment une boucle A→B→A :", loops.count())
print("Patients concernés :", loops.select("patient_id").distinct().count())
loops.groupBy("patient_id").count().orderBy(F.desc("count")).show(8)

# recoupement avec la règle 1
r1 = tr.groupBy("patient_id").count().filter("count >= 4").select("patient_id")
both = loops.select("patient_id").distinct().join(r1, "patient_id")
print("Patients à la fois en boucle et avec >= 4 transferts :", both.count())

Transferts qui referment une boucle A→B→A : 142


Patients concernés : 57


+----------+-----+
|patient_id|count|
+----------+-----+
|   P-01570|    3|
|   P-03058|    3|
|   P-01583|    3|
|   P-00129|    3|
|   P-01922|    3|
|   P-00319|    3|
|   P-02130|    3|
|   P-00891|    3|
+----------+-----+
only showing top 8 rows



[Stage 79:>                                                         (0 + 1) / 1]

Patients à la fois en boucle et avec >= 4 transferts : 52


In [7]:
fac_info = (verts.filter("type = 'FACILITY'")
            .select(F.col("id").alias("fid"), F.col("hospital_group").alias("g"), F.col("name")))

src_i = fac_info.select(F.col("fid").alias("src"), F.col("g").alias("src_group"))
dst_i = fac_info.select(F.col("fid").alias("dst"), F.col("g").alias("dst_group"), F.col("name").alias("dst_name"))

t = tr.join(src_i, "src").join(dst_i, "dst")
inter = t.filter("src_group != dst_group")
viol = inter.filter(~F.col("dst_name").startswith("CHU"))

print("Transferts inter-groupes :", inter.count(), "(attendu 138)")
print("Dont vers un non-CHU (violations) :", viol.count())
viol.select("patient_id", "visit_id", "src", "dst", "src_group", "dst_group", "event_time") \
    .orderBy("event_time").show(15, truncate=False)

[Stage 84:===========================================>              (3 + 1) / 4]

Transferts inter-groupes : 138 (attendu 138)


Dont vers un non-CHU (violations) : 1


+----------+---------+-----+-----+--------------+---------+-------------------+
|patient_id|visit_id |src  |dst  |src_group     |dst_group|event_time         |
+----------+---------+-----+-----+--------------+---------+-------------------+
|P-04788   |V-0005629|F-012|F-026|GHT-Atlantique|GHT-Sud  |2026-10-18 17:03:45|
+----------+---------+-----+-----+--------------+---------+-------------------+



In [8]:
recv = (t.groupBy("dst", "dst_group")
          .agg(F.count("*").alias("recus"),
               F.sum(F.when(F.col("src_group") == F.col("dst_group"), 1).otherwise(0)).alias("intra"))
          .withColumn("part_intra", F.round(F.col("intra") / F.col("recus"), 3)))

print("Établissements dont moins de 80 % des transferts reçus viennent du groupe :")
recv.filter("part_intra < 0.8").orderBy("part_intra").show(15, truncate=False)

print("Distribution de la part intra-groupe :")
recv.agg(F.min("part_intra").alias("min"), F.round(F.avg("part_intra"), 3).alias("moy"),
         F.max("part_intra").alias("max")).show()

Établissements dont moins de 80 % des transferts reçus viennent du groupe :


+-----+--------------+-----+-----+----------+
|dst  |dst_group     |recus|intra|part_intra|
+-----+--------------+-----+-----+----------+
|F-031|GHT-Nord      |122  |73   |0.598     |
|F-021|GHT-Sud       |130  |85   |0.654     |
|F-011|GHT-Atlantique|102  |78   |0.765     |
+-----+--------------+-----+-----+----------+

Distribution de la part intra-groupe :


[Stage 100:============================>                            (2 + 2) / 4]

+-----+-----+---+
|  min|  moy|max|
+-----+-----+---+
|0.598|0.969|1.0|
+-----+-----+---+



In [9]:
base_out = base + "anomalies/"

# groupe du patient = groupe de l'établissement source du transfert
src_grp = t.select("patient_id", "src_group").dropDuplicates(["patient_id"]) \
           .withColumnRenamed("src_group", "hospital_group")

r1 = (by_p.select("patient_id", "nb_transferts", "chemin")
        .join(src_grp, "patient_id", "left")
        .withColumn("regle", F.lit("R1_ge4_transferts")))

r2 = (loops.select("patient_id", "event_time", "src", "dst")
        .join(src_grp, "patient_id", "left")
        .withColumn("regle", F.lit("R2_boucle_ABA")))

r3 = (viol.select("patient_id", "visit_id", "src", "dst", "event_time",
                  F.col("src_group").alias("hospital_group"))
        .withColumn("regle", F.lit("R3_inter_groupe_non_CHU")))

r4 = (recv.filter("part_intra < 0.8")
        .select(F.col("dst").alias("facility_id"), F.col("dst_group").alias("hospital_group"),
                "recus", "intra", "part_intra")
        .withColumn("regle", F.lit("R4_part_intra_faible")))

for name, df in [("r1_patients_ge4", r1), ("r2_boucles", r2),
                 ("r3_violations", r3), ("r4_poles", r4)]:
    df.write.mode("overwrite").partitionBy("hospital_group").parquet(base_out + name)

for name in ["r1_patients_ge4", "r2_boucles", "r3_violations", "r4_poles"]:
    d = spark.read.parquet(base_out + name)
    print(name, ":", d.count(), "lignes")
    d.groupBy("hospital_group").count().orderBy("hospital_group").show()

r1_patients_ge4 : 79 lignes


+--------------+-----+
|hospital_group|count|
+--------------+-----+
|GHT-Atlantique|   41|
|      GHT-Nord|   19|
|       GHT-Sud|   19|
+--------------+-----+

r2_boucles : 142 lignes


+--------------+-----+
|hospital_group|count|
+--------------+-----+
|GHT-Atlantique|   62|
|      GHT-Nord|   38|
|       GHT-Sud|   42|
+--------------+-----+

r3_violations : 1 lignes
+--------------+-----+
|hospital_group|count|
+--------------+-----+
|GHT-Atlantique|    1|
+--------------+-----+

r4_poles : 3 lignes


[Stage 154:======================================>                  (2 + 1) / 3]

+--------------+-----+
|hospital_group|count|
+--------------+-----+
|GHT-Atlantique|    1|
|      GHT-Nord|    1|
|       GHT-Sud|    1|
+--------------+-----+



In [10]:
metrics = (gf.select(F.col("id").alias("facility_id"), "name", "city",
                     F.col("outDegree").alias("gf_out"), F.col("inDegree").alias("gf_in"))
             .join(gx.select("facility_id", "hospital_group", "degree", "weighted_degree",
                             "pagerank", "component"), "facility_id"))

metrics.write.mode("overwrite").partitionBy("hospital_group") \
       .parquet(base + "metrics_par_etablissement")

m = spark.read.parquet(base + "metrics_par_etablissement")
print("lignes (attendu 40) :", m.count())
m.orderBy(F.desc("pagerank")).select("facility_id", "name", "hospital_group", "gf_out", "gf_in",
    "weighted_degree", F.round("pagerank", 3).alias("pagerank"), "component").show(10, truncate=False)

lignes (attendu 40) : 40


+-----------+--------------------------------+--------------+------+-----+---------------+--------+---------+
|facility_id|name                            |hospital_group|gf_out|gf_in|weighted_degree|pagerank|component|
+-----------+--------------------------------+--------------+------+-----+---------------+--------+---------+
|F-013      |Hôpital Régional Sud Rabat      |GHT-Atlantique|53    |58   |106            |1.805   |1        |
|F-021      |CHU Marrakech                   |GHT-Sud       |97    |130  |173            |1.762   |21       |
|F-023      |Hôpital Régional Sud Marrakech  |GHT-Sud       |62    |55   |112            |1.743   |21       |
|F-011      |CHU Rabat                       |GHT-Atlantique|111   |102  |174            |1.733   |1        |
|F-012      |Hôpital Régional Nord Rabat     |GHT-Atlantique|51    |48   |90             |1.617   |1        |
|F-002      |Hôpital Régional Nord Casablanca|GHT-Atlantique|36    |44   |77             |1.586   |1        |
|F-001    

In [11]:
import networkx as nx
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

grp = "GHT-Sud"
w_pd = (spark.read.parquet(base + "input/transfer_weights")
        .filter(f"src_group = '{grp}' AND dst_group = '{grp}'").toPandas())
pr_pd = (spark.read.parquet(base + "graphx/metrics")
         .filter(f"hospital_group = '{grp}'").select("facility_id", "pagerank").toPandas())
pr = dict(zip(pr_pd.facility_id, pr_pd.pagerank))

G = nx.DiGraph()
for _, r in w_pd.iterrows():
    G.add_edge(r.src, r.dst, weight=int(r.weight))

pos = nx.circular_layout(G)
fig, ax = plt.subplots(figsize=(9, 8))
nx.draw_networkx_nodes(G, pos, node_size=[pr[n] * 900 for n in G.nodes], node_color="#4C9BE8", ax=ax)
nx.draw_networkx_labels(G, pos, font_size=9, ax=ax)
nx.draw_networkx_edges(G, pos, width=[G[u][v]["weight"] / 3 for u, v in G.edges],
                       alpha=0.4, arrowsize=10, connectionstyle="arc3,rad=0.1", ax=ax)
ax.set_title(f"Transferts intra-groupe {grp} (arêtes = nb de transferts, nœuds = PageRank)")
ax.axis("off")
fig.savefig("/workspace/data/graphe_ght_sud.png", dpi=150, bbox_inches="tight")
print("noeuds :", G.number_of_nodes(), "| arêtes :", G.number_of_edges())

noeuds : 10 | arêtes : 50


In [12]:
spark.stop()